<a href="https://colab.research.google.com/github/a1ephh/pytorch101/blob/main/learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,),(0.3081,))
])

train_dataset = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset= torchvision.datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader= torch.utils.data.DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader= torch.utils.data.DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

print(f"Dataset loaded! Training images: {len(train_dataset)}, Testing images: {len(test_dataset)}")


Dataset loaded! Training images: 60000, Testing images: 10000


In [ ]:
class BrainNetwork(nn.Module):
  def __init__(self):
    super(BrainNetwork, self).__init__()

    self.layer1=nn.Linear(28*28, 128)

    self.relu = nn.ReLU()

    self.layer2=nn.Linear(128, 10)

  def forward(self,x):
    x=x.reshape(-1, 28*28)
    x=self.layer1(x)
    x=self.relu(x)
    x=self.layer2(x)
    return x


model = BrainNetwork()
print(model)

BrainNetwork(
  (layer1): Linear(in_features=784, out_features=128, bias=True)
  (relu): ReLU()
  (layer2): Linear(in_features=128, out_features=10, bias=True)
)


In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(model.parameters(), lr=0.01)

In [ ]:
epochs = 3

for epoch in range(epochs):
  running_loss=0.0

  for images, labels in train_loader:
    optimizer.zero_grad()

    outputs = model(images)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    running_loss += loss.item()

  print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader)}")
print("Training finished!")

Epoch 1/3, Loss: 0.5752839234147245
Epoch 2/3, Loss: 0.29781783116397575
Epoch 3/3, Loss: 0.2501448053461529
Training finished!


In [ ]:
weight_matrix = model.layer1.weight.data
gradient_matrix= model.layer1.weight.grad

print("--- LAYER 1 PARAMETER INSPECTION ---")
print(f"Physical shape of the weight matrix: {weight_matrix.shape}")
print(f"Physical shape of the gradient matrix: {gradient_matrix.shape}\n")

print("A tiny slice of the actual trained weights:")
print(weight_matrix[:3, :3])

# Let's check the gradients left over from the very last batch
print("\nA tiny slice of the remaining gradients in the '.grad' pocket:")
print(gradient_matrix[:3, :3])

--- LAYER 1 PARAMETER INSPECTION ---
Physical shape of the weight matrix: torch.Size([128, 784])
Physical shape of the gradient matrix: torch.Size([128, 784])

A tiny slice of the actual trained weights:
tensor([[ 0.0009, -0.0112, -0.0289],
        [-0.0068,  0.0091,  0.0007],
        [-0.0250,  0.0162,  0.0067]])

A tiny slice of the remaining gradients in the '.grad' pocket:
tensor([[ 4.7233e-04,  4.7233e-04,  4.7233e-04],
        [ 1.7925e-03,  1.7925e-03,  1.7925e-03],
        [-4.0254e-05, -4.0254e-05, -4.0254e-05]])


In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Accuracy of the network on the 10,000 test images: {accuracy:.2f}%")

Accuracy of the network on the 10,000 test images: 93.56%


In [9]:
import torch

def get_split_mnist(dataset, allowed_digits, batch_size, shuffle):
    mask = torch.tensor([label in allowed_digits for _, label in dataset])
    true_indices = torch.where(mask)[0]

    filtered_subset = torch.utils.data.Subset(dataset, true_indices)
    return torch.utils.data.DataLoader(filtered_subset, batch_size=batch_size, shuffle=shuffle)

print("Blueprint function 'get_split_mnist' successfully created!")

Blueprint function 'get_split_mnist' successfully created!


In [10]:
task1_digits = [0, 1, 2, 3, 4]
task2_digits = [5, 6, 7, 8, 9]

task2_train_loader = get_split_mnist(train_dataset, task2_digits, batch_size=64, shuffle=True)
task2_test_loader = get_split_mnist(test_dataset, task2_digits, batch_size=64, shuffle=False)

task1_test_loader = get_split_mnist(test_dataset, task1_digits, batch_size=64, shuffle=False)

print(f"Initialization complete!")
print(f"-> task2_train_loader is alive with {len(task2_train_loader)} batches of high digits (5-9).")
print(f"-> task1_test_loader is alive with {len(task1_test_loader)} batches of low digits (0-4).")

Initialization complete!
-> task2_train_loader is alive with 460 batches of high digits (5-9).
-> task1_test_loader is alive with 81 batches of low digits (0-4).


In [12]:
epochs_task2 = 2

print("Training exclusively on digits 5-9...")
for epoch in range(epochs_task2):
    running_loss = 0.0
    for images, labels in task2_train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Task 2 - Epoch {epoch+1}/{epochs_task2} - Loss: {running_loss/len(task2_train_loader):.4f}")

print("Training on Task 2 finished!\n")


print("Testing the model's memory on Task 1 (Digits 0-4)...")
model.eval()
correct_t1 = 0
total_t1 = 0

with torch.no_grad():
    for images, labels in task1_test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total_t1 += labels.size(0)
        correct_t1 += (predicted == labels).sum().item()

accuracy_t1 = 100 * correct_t1 / total_t1
print(f"Final Accuracy on Task 1 (Digits 0-4): {accuracy_t1:.2f}%")

Training exclusively on digits 5-9...
Task 2 - Epoch 1/2 - Loss: 0.1058
Task 2 - Epoch 2/2 - Loss: 0.0970
Training on Task 2 finished!

Testing the model's memory on Task 1 (Digits 0-4)...
Final Accuracy on Task 1 (Digits 0-4): 15.86%
